# Tutorial for Training/Deploying the Physical CartPole Demo

This notebook follows the demo-hls4ml-25 README (Steps 1–7)

> Use the navigation links below to jump around quickly.


## Navigation
- [Step 0 — Notebook prerequisites](#step-0-notebook-prerequisites)
- [Step 1 — Environment Setup](#step-1-environment-setup)
- [Step 2 — Training Neural Network Controller](#step-2-training-neural-network-controller)
- [Step 3 — Running the Cartpole Simulator](#step-3-running-the-cartpole-simulator)
<!--
- [Step 4 — hls4ml Conversion](#step-4--conversion-of-neural-network-controller-using-hls4ml)
- [Step 5 — Testing on PC / Software Control](#step-5--testing-model-on-pc--running-model-via-pcsoftware-to-control-cartpole)
- [Step 6 — Implementation (Vivado/Vitis)](#step-6--implementation)
- [Step 7 — SD Card + FPGA Boot](#step-7--load-image-on-sd-card-and-onto-fpga)
-->


## Step 0: Notebook prerequisites 

- Please make sure you have Conda installed
    - Refer to [this article](https://docs.conda.io/projects/conda/en/latest/user-guide/install/index.html) for instructions on setting it up
- Ensure you have access to these programs
    - Vivado 2020.1 
    - Vitis 2020.1


## Step 1: Environment Setup

### Conda environment
In your terminal run these
```bash
conda create -n physical_cartpole python=3.9
conda activate physical_cartpole
```
### Install Packages
These are just the necessary packages for training, we will install the GUI/Simulation packages later
<!-- these are the gui packages: # These need PyQt6 which I dont have rn (for the GUI)
## -e ./Driver/CartPoleSimulation/SI_Toolkit/
## %pip install watchdog pydot graphviz PyQt6 -->
**Lab server note:** if your environment is pre-configured, you can skip this step.


In [ ]:
%pip install -r requirements.txt 
%pip install watchdog pydot graphviz PyQt6


### 1.1 Choose experiment + model name & set paths for later

- `CARTPOLE_EXPERIMENT_NAME` (default: `Experiment-1`)
- `CARTPOLE_NET_NAME` (default shown below)


In [ ]:
import os, shutil
import sys

from pathlib import Path

# Gets a couple of paths to be referenced later on (assumes notebook is inside the repo)
REPO = Path.cwd().resolve()
WORKSPACE = REPO / "Driver" / "CartPoleSimulation" / "SI_Toolkit_ASF" / "Experiments"
SI_ASF = REPO / "Driver" / "CartPoleSimulation" / "SI_Toolkit_ASF"


# Sets the Experiment name 
EXPERIMENT_NAME = os.environ.get("CARTPOLE_EXPERIMENT_NAME", "Experiment-1")

# Sets the name for the neural network model
NET_NAME = os.environ.get("CARTPOLE_NET_NAME", "Dense-7IN-32H1-32H2-1OUT-0") 

# Path to seed (template) experiment that is provided
SEED_EXPERIMENT = REPO / "Experiment-1"

# Path to active experiment workspace used by SI_Toolkit
ACTIVE_EXPERIMENT = WORKSPACE / EXPERIMENT_NAME

# Create a path to the simulation 
SIM = REPO / "Driver" / "CartPoleSimulation"

# Prepend SI_Toolkit/src so `import SI_Toolkit...`
si_src = SIM / "SI_Toolkit" / "src"
if str(si_src) not in sys.path:
    sys.path.insert(0, str(si_src))

# Also prepend the simulator root so `SI_Toolkit_ASF` resolves cleanly when running from the notebook
if str(SIM) not in sys.path:
    sys.path.insert(0, str(SIM))

# Echo resolved paths and selected model for verification
print("SEED_EXPERIMENT:", SEED_EXPERIMENT)
print("ACTIVE_EXPERIMENT:", ACTIVE_EXPERIMENT)
print("NET_NAME:", NET_NAME)
print("SIM =", SIM)
print("SI_Toolkit src =", si_src)

### 1.2 Move/copy seed experiment into the SI_Toolkit experiments folder

This shell scripts effectively ensure the experiment lives under `SI_Toolkit_ASF/Experiments/`.
We do a safe copy **only if missing**, to avoid overwriting trained artifacts.


In [ ]:
import shutil

if not ACTIVE_EXPERIMENT.exists():
    shutil.copytree(SEED_EXPERIMENT, ACTIVE_EXPERIMENT)
    print("Copied ./Experiment-1 →", ACTIVE_EXPERIMENT)
else:
    print("Active experiment already exists: not overwriting")

## Step 2: Training Neural Network Controller

### Dataset location
The Seed dataset is initially located at: `./Experiment-1` in the root directory which contains:
- recorded trajectories (CSV)
- a known good model configuration

Now there are **two paths**:

### Path A: Use precomputed model
- Use a pre-trained model from the seed folder `../Experiment-1/Models` (chosen in step 1)
- Skip to the Simulation (Step 3) 

### Path B: Train the neural network
- Follow the next steps

### 2.1 Training Configuration (config_training.yml)

Run this cell to see the current training configurations in:
`Driver/CartPoleSimulation/SI_Toolkit_ASF/config_training.yml`

Please refer to the [training_configurations_walkthrough](training_configurations_walkthrough.ipynb) notebook for more details on the training configurations

In [ ]:
import yaml
from pprint import pprint

with open("../physical-cartpole/Driver/CartPoleSimulation/SI_Toolkit_ASF/config_training.yml", "r") as f:
    cfg = yaml.safe_load(f)

pprint(cfg)

Below is an **optional** cell that updates the yaml's experiment path if you changed it

In [ ]:
import yaml

cfg_path = SI_ASF / "config_training.yml"
cfg = yaml.safe_load(cfg_path.read_text())

def set_experiment_path(cfg_obj, new_path: str) -> bool:
    if isinstance(cfg_obj, dict):
        if "paths" in cfg_obj and isinstance(cfg_obj["paths"], dict) and "path_to_experiment" in cfg_obj["paths"]:
            cfg_obj["paths"]["path_to_experiment"] = new_path
            return True
        if "PATH_TO_EXPERIMENT" in cfg_obj:
            cfg_obj["PATH_TO_EXPERIMENT"] = new_path
            return True
    return False

ok = set_experiment_path(cfg, str(ACTIVE_EXPERIMENT))
if ok:
    cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
    print("Updated config_training.yml to use:", ACTIVE_EXPERIMENT)
else:
    print("Could not locate experiment path key; set it manually to:", ACTIVE_EXPERIMENT)

### 2.2 Data normalization

Neural networks train much more reliably when each feature is on a comparable numeric scale.

In this project, **normalization statistics are computed from the `Train/` CSVs** inside the active experiment folder.  
Those statistics are saved to `NormalizationInfo/` and then reused consistently for:

- training 
- simulation inference
- FPGA/HLS deployment

The function that does this is `SI_Toolkit.load_and_normalize.calculate_normalization_info(...)`.


In [ ]:
# Load the training configuration
import yaml

cfg_path = SI_ASF / "config_training.yml"
cfg = yaml.safe_load(cfg_path.read_text())

# The two path keys
print("PATH_TO_EXPERIMENT_FOLDERS =", cfg["paths"]["PATH_TO_EXPERIMENT_FOLDERS"])
print("path_to_experiment         =", cfg["paths"]["path_to_experiment"])
print("DATA_FOLDER                =", cfg["paths"]["DATA_FOLDER"])

train_dir = Path(cfg["paths"]["PATH_TO_EXPERIMENT_FOLDERS"]) / cfg["paths"]["path_to_experiment"] / cfg["paths"]["DATA_FOLDER"] / "Train"
print("\nTrain directory (resolved):", train_dir.resolve())


In [ ]:
# Identify the CSVs that are used for normalization
import glob

train_csvs = sorted(glob.glob(str(train_dir / "*.csv")))
print(f"Found {len(train_csvs)} training CSV files.")
for p in train_csvs[:10]:
    print(" -", p)
if len(train_csvs) > 10:
    print(f" ... ({len(train_csvs)-10} more)")


In [ ]:
from SI_Toolkit.load_and_normalize import get_paths_to_datafiles, load_data

# Build list of CSV paths
train_paths = get_paths_to_datafiles(str(train_dir))
val_dir = train_dir.parent / "Validation"
test_dir = train_dir.parent / "Test"

# Load CSVs into DataFrames
training_dfs = load_data(train_paths)

validation_dfs = load_data(get_paths_to_datafiles(str(val_dir)))
test_dfs       = load_data(get_paths_to_datafiles(str(test_dir)))

print("Loaded training files:", len(training_dfs))
print("Example columns:", training_dfs[0].columns.tolist())

#### 2.2.1 Compute normalization statistics + write `NormalizationInfo/NI_*.csv`

By default the function:
- concatenates all Train CSVs into one DataFrame
- drops a `time` column (if present)
- computes per-feature **mean/std/min/max**
- optionally applies a **user correction hook** from `SI_Toolkit_ASF/ToolkitCustomization/...`
- writes a timestamped `NI_YYYY-MM-DD_HH-MM-SS.csv`
- optionally saves histogram PNGs per feature


In [ ]:
# Run the normalization (preprocessing) step
from SI_Toolkit.load_and_normalize import calculate_normalization_info

# Keep histograms ON for now, optionally turn OFF for speed in automated runs
df_norm_info, norm_csv_path = calculate_normalization_info(
    config=cfg,
    plot_histograms=True,
    user_correction=False, # was true, I changed this?
)

print("Wrote normalization CSV:", norm_csv_path)
display(df_norm_info)


#### 2.2.2 How to read `df_norm_info`

`df_norm_info` is indexed by the statistic (`mean`, `std`, `min`, `max`).  
Each column is a feature from your training CSVs (excluding `time`).

Typical z-score normalization uses:

- **normalize**:  \(x_{norm} = (x - \mu) / \sigma\)
- **denormalize**: \(x = x_{norm} \cdot \sigma + \mu\)

The exact columns used as inputs/targets are determined by your training config and the model wrapper,
but the statistics come from the raw CSV columns.


#### 2.2.3 Where the histograms went

If `plot_histograms=True`, the function saves one histogram per feature to:

`.../NormalizationInfo/histograms/<feature>.png`

In [ ]:
# Show a couple histogram images (if they were generated)
from pathlib import Path

hist_dir = Path(norm_csv_path).parent / "histograms"
print("Histogram dir:", hist_dir)

if hist_dir.exists():
    pngs = sorted(hist_dir.glob("*.png"))
    print("Found", len(pngs), "histograms.")
else:
    print("No histograms folder found (plot_histograms may be False).")


### 2.3 Train the neural network
At this point, we have written a normalization file from the **Train** split. Now we go through the training flow.

Training is config driven, uses the normalization statistics computed above, and writes a fully reproducible model folder under `Models/`.

#### 2.3.1 Load training arguments (YAML + CLI overrides)

Training is config driven: `args()` loads `config_training.yml` defaults and applies any CLI overrides.
We print the resolved paths and split files so you can confirm the **active experiment**.


In [ ]:
## Set the correct path to run training 
from pathlib import Path
import os

# Find the repo root by walking up until we see "Driver"
here = Path.cwd().resolve()

# If you're already in the repo root, this will work:
sim_dir = (here / "Driver" / "CartPoleSimulation").resolve()

# If you're inside a subdir, walk up a few levels to find the repo root:
if not sim_dir.exists():
    for p in [here] + list(here.parents):
        candidate = (p / "Driver" / "CartPoleSimulation")
        if candidate.exists():
            sim_dir = candidate.resolve()
            break

print("Setting working directory to:", sim_dir)
os.chdir(sim_dir)

print("Now cwd =", Path.cwd())
print("Config exists? ", (Path("SI_Toolkit_ASF/config_training.yml")).exists())


In [ ]:
from SI_Toolkit.Functions.General.load_parameters_for_training import args 
from SI_Toolkit.Functions.General.Initialization import set_seed

# This also fixes issues with flags that I could also fix eventually
# Save Jupyter argv 
_argv_backup = sys.argv.copy()

# Make argparse think we're running with no CLI args
sys.argv = [sys.argv[0]]

# This parses the yaml file for our training details. 
# here we can change things like batch size and epochs
a = args()

# Sets random seed
set_seed(a)

# Restore argv
sys.argv = _argv_backup
print("Key training settings:")
for k in ["net_name", "library", "path_to_models", "training_files", "validation_files", "test_files", "config_path"]:
    if hasattr(a, k):
        print(f"  {k}: {getattr(a, k)}")


#### 2.3.2 Instantiate the model and inspect data

`get_net(a)` builds the network and a `net_info` object that describes naming, paths, and whether normalization is enabled. `create_full_name(...)` generates the run folder name.


In [ ]:
from SI_Toolkit.Functions.General.Initialization import get_net, create_full_name

net, net_info = get_net(a)
create_full_name(net_info, a.path_to_models)

print("Model full name:", net_info.net_full_name)
print("Backend library:", net_info.library)
print("Will normalize:", getattr(net_info, "normalize", None))
print("Models root folder:", a.path_to_models)
print("This run will be saved under:", f"{a.path_to_models}/{net_info.net_full_name}")


#### 2.3.3 Load normalization info

This is why normalization must run first: training needs the mean/std (and related vectors) to scale features.
These vectors are also reused later for consistent inference (including HLS/firmware).


In [ ]:
from SI_Toolkit.Functions.General.Initialization import get_norm_info_for_net
from SI_Toolkit.Functions.General.Normalising import write_out_normalization_vectors

normalization_info = get_norm_info_for_net(net_info, files_for_normalization=a.training_files)
write_out_normalization_vectors(normalization_info, net_info)

print("Normalization info loaded and vectors written for:", net_info.net_full_name)

#### 2.3.4 Load Train/Validate/Test CSVs 

Here we resolve file lists for each split, load them as DataFrames, and print basic counts. This connects CSV logs to the training tensors.


In [ ]:
from SI_Toolkit.load_and_normalize import load_data, get_paths_to_datafiles
from SI_Toolkit.Functions.General.Initialization import create_log_file
import os

train_paths = get_paths_to_datafiles(a.training_files)
val_paths   = get_paths_to_datafiles(a.validation_files)
test_paths  = get_paths_to_datafiles(a.test_files)

training_dfs   = load_data(train_paths)
validation_dfs = load_data(val_paths)
test_dfs       = load_data(test_paths)

run_dir = os.path.join(a.path_to_models, net_info.net_full_name)
os.makedirs(run_dir, exist_ok=True)

create_log_file(net_info, a, training_dfs)

def nrows(dfs):
    return sum(len(df) for df in dfs) if isinstance(dfs, list) else len(dfs)

print("Train files:", len(train_paths), "rows:", nrows(training_dfs))
print("Val files:  ", len(val_paths),   "rows:", nrows(validation_dfs))
print("Test files: ", len(test_paths),  "rows:", nrows(test_dfs))

example_cols = training_dfs[0].columns if isinstance(training_dfs, list) else training_dfs.columns
print("Example columns:", list(example_cols))


#### 2.3.5 Adding Previous Control Input (Q_applied_-1)

This step augments each dataset with a **one-timestep delayed control input**.  
For every DataFrame, we sort by time, shift `Q_applied` by one step to create `Q_applied_-1`, and drop the first row (which has no previous value).  

This provides the model access to the **previous control action** and matches the network’s expected inputs.


In [ ]:
def add_prev_Q_applied(dfs, time_col="time"):
    for df in dfs:
        if "Q_applied_-1" in df.columns:
            continue

        # Ensure sorted by time just in case
        if time_col in df.columns:
            df.sort_values(time_col, inplace=True)

        df["Q_applied_-1"] = df["Q_applied"].shift(1)

        # First row has no previous value → drop it
        df.dropna(subset=["Q_applied_-1"], inplace=True)
        df.reset_index(drop=True, inplace=True)

add_prev_Q_applied(training_dfs)
add_prev_Q_applied(validation_dfs)
add_prev_Q_applied(test_dfs)

# Verify
df0 = training_dfs[0]
# print("Missing now:", (set(net_info.inputs) | set(net_info.outputs)) - set(df0.columns))
print(df0[["time", "Q_applied", "Q_applied_-1", "Q_calculated"]].head(5))


#### 2.3.6 Run the training loop

This is the real training loop used by `train_network()`: it calls `Training.train_network_core(...)`.
It returns loss curves used to generate the training plot.

**Note:** Training can take alot of time


In [ ]:
if net_info.library == "TF":
    import SI_Toolkit.Functions.TF.Training as Training
else:
    import SI_Toolkit.Functions.Pytorch.Training as Training

from SI_Toolkit.Functions.General.TerminalContentManager import TerminalContentManager

with TerminalContentManager(os.path.join(run_dir, "terminal_output.txt")):
    loss, val_loss, post_epoch_loss = Training.train_network_core(
        net, net_info,
        training_dfs,
        validation_dfs,
        test_dfs,
        normalization_info,
        a
    )

print("Final validation loss:", val_loss[-1])


#### 2.5.6 Plot losses and point to saved artifacts

After training finishes, the run folder under `Models/` contains:
- the trained model/checkpoints
- `terminal_output.txt` (captured outputs)
- copied config + training script
- `training_curve.png`


In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

plt.figure()
plt.plot(loss, label="train")
plt.plot(val_loss, label="val")
plt.yscale("log")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend()
plt.title(net_info.net_full_name)
plt.savefig(os.path.join(net_info.path_to_net, "training_curve.png"))
plt.show()

model_dir = Path(a.path_to_models) / net_info.net_full_name
print("Model artifacts saved in:", model_dir)
print("Look for: training_curve.png, terminal_output.txt, config copy, checkpoints/models")


## Step 3: Running the Cartpole Simulator
In this section, we will primarily focus on using the Cartpole Simulator to test the performance of trained neural network controllers. This simulator not only allows for performance evaluation but can also be used to generate new datasets for further training. Here, we will describe how to effectively use the simulator to assess your model's capabilities.

### 3.1 Choose the Model
   - Run the program with the desired model name as an argument to select a specific trained model. If no model name is provided, the default pre-trained model will be used.
   - Available model names can be found in the following folder:
     ```
     Driver/CartPoleSimulation/SI_Toolkit_ASF/Experiments/Experiment-1/Models
     ```

### 3.2 Run the GUI
- Make sure you have access to a display variable


In [ ]:
import subprocess
from pathlib import Path

# Find repo root by walking upward until step3.sh is found
cwd = Path.cwd().resolve()
repo_root = None

for parent in [cwd] + list(cwd.parents):
    if (parent / "step3.sh").exists():
        repo_root = parent
        break

if repo_root is None:
    raise FileNotFoundError("Could not find step3.sh in any parent directory")

script = repo_root / "step3.sh"
script.chmod(script.stat().st_mode | 0o111)

args = ["bash", str(script)]
if NET_NAME:
    args.append(NET_NAME)

print("Notebook CWD:", cwd)
print("Repo root:", repo_root)
print("Running:", " ".join(args))

# Run from repo root so relative paths resolve
subprocess.run(args, cwd=str(repo_root), check=True)
